In [1]:
from strange_loop import CodexBackend, Runner, flow, stage


@stage
def clean(text):
    return text.strip()

@stage
def word_count(text):
    return len(text.split())

In [2]:
@flow
async def count_words(ctx, *, text):
    cleaned = await clean(text=text)
    count = await word_count(text=cleaned)
    return count


In [6]:

runner = Runner(backend=CodexBackend())
count = await runner.run(count_words, text="  two words  ")
print(runner.store.read(count))  # 2



2


In [7]:
count

Ref(id=3)

In [8]:
cleaned = runner.store.inputs(count)["text"]
original = runner.store.inputs(cleaned)["text"]

for ref in (original, cleaned, count):
    print(ref, repr(runner.store.read(ref)), runner.store.inputs(ref))


Ref(id=1) '  two words  ' {}
Ref(id=2) 'two words' {'text': Ref(id=1)}
Ref(id=3) 2 {'text': Ref(id=2)}


In [9]:
for row in runner.store.db.execute("SELECT id, slot, value, bindings FROM revisions ORDER BY id"):
    print(row)


(1, '["aae4212aa3df4a5b82f10652043f1fc5", "count_words", "text"]', '"  two words  "', '{}')
(2, '["aae4212aa3df4a5b82f10652043f1fc5", "count_words", "clean", 0, ""]', '"two words"', '{"text": 1}')
(3, '["aae4212aa3df4a5b82f10652043f1fc5", "count_words", "word_count", 0, ""]', '2', '{"text": 2}')
